# Build Your Own Benchmark: finance MCQ example

This notebook demonstrates the current Nemotron BYOB step structure. It does not import MCQ pipeline internals directly. Instead, it prepares an example corpus, writes a working config from `src/nemotron/steps/byob/config/default.yaml`, and runs the public CLI surface: `nemotron byob`.

The generated benchmark keeps the MMLU-Pro-style BYOB schema documented in `src/nemotron/steps/byob/references/benchmark-schema.md`.

## What this notebook creates

- `assets/wiki_finance/*.txt`: finance-related source documents.
- `config/finance_wiki.yaml`: a working MCQ config derived from the packaged BYOB default config.
- `outputs/finance_wiki_mcq/seed.parquet`: BYOB seed data after `prepare`.
- `outputs/finance_wiki_mcq/benchmark.parquet`: final MCQ benchmark after `generate`.
- `config/finance_wiki_translate.yaml`: optional translation config derived from the packaged BYOB translation config.

The run cells default to command preview mode so the notebook does not accidentally spend model quota.

## 1. Locate the repository and BYOB step assets

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

import yaml


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src/nemotron/steps/byob").is_dir():
            return candidate
    raise RuntimeError("Could not find the Nemotron repository root from the current working directory.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_DIR = REPO_ROOT / "src"
EXAMPLE_DIR = REPO_ROOT / "use-case-examples" / "build-your-own-benchmark"
STEP_CONFIG_DIR = SRC_DIR / "nemotron" / "steps" / "byob" / "config"
STEP_DEFAULT_CONFIG = STEP_CONFIG_DIR / "default.yaml"
STEP_TRANSLATE_CONFIG = STEP_CONFIG_DIR / "translate.yaml"

ASSETS_DIR = EXAMPLE_DIR / "assets" / "wiki_finance"
CONFIG_DIR = EXAMPLE_DIR / "config"
OUTPUT_DIR = EXAMPLE_DIR / "outputs"
WORKING_CONFIG = CONFIG_DIR / "finance_wiki.yaml"
TRANSLATION_CONFIG = CONFIG_DIR / "finance_wiki_translate.yaml"

for path in (ASSETS_DIR, CONFIG_DIR, OUTPUT_DIR):
    path.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Repository root:", REPO_ROOT)
print("Packaged BYOB config:", STEP_DEFAULT_CONFIG)
print("Example corpus dir:", ASSETS_DIR)
print("Working config:", WORKING_CONFIG)

## 1b. Sanity check: kernel env and Ray cluster

BYOB runs nemo-curator inside Ray workers. If Jupyter is using a kernel that doesn't have the `byob` extra installed, or if a Ray cluster from a *different* project venv is already running, you'll later see `ModuleNotFoundError: No module named 'nemo_curator'` coming from inside Ray. Catch it now instead.

If anything below fails, see the **Setup** section in `README.md` and run `ray stop --force` if a stale cluster is detected.

In [ ]:
import importlib
import shutil

print("Kernel python:", sys.executable)
print("Project venv :", REPO_ROOT / ".venv" / "bin" / "python")

expected = (REPO_ROOT / ".venv" / "bin" / "python").resolve()
actual = Path(sys.executable).resolve()
if actual != expected:
    print(
        f"WARNING: kernel python is not {expected}. "
        "Start jupyter from the project root with `uv run --extra byob jupyter lab` so the kernel uses .venv."
    )

required = ["nemo_curator", "ray", "sentence_transformers", "cudf", "cuml"]
missing = []
for mod in required:
    try:
        m = importlib.import_module(mod)
        print(f"  ok  {mod:25s} {getattr(m, '__version__', '?'):>10s}  {getattr(m, '__file__', '')}")
    except Exception as exc:
        missing.append(mod)
        print(f"  FAIL {mod:25s} -> {exc.__class__.__name__}: {exc}")

if missing:
    raise RuntimeError(
        f"Missing modules in kernel: {missing}. "
        "From the repo root run: `uv sync --extra byob`, then restart this kernel."
    )

# Detect ancestor `uv run` invocation. Ray's uv_runtime_env_hook reconstructs
# the *driver's* `uv run` command for workers, capturing only flags that
# appeared before the command. If `--extra byob` is missing here, workers
# will fail with `ModuleNotFoundError: nemo_curator` deep inside a Ray task.
try:
    import psutil  # ships with ray

    uv_cmdline = None
    for parent in psutil.Process().parents():
        try:
            cmd = parent.cmdline()
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            continue
        if len(cmd) > 1 and Path(cmd[0]).name == "uv" and cmd[1] == "run":
            uv_cmdline = cmd
            break

    if uv_cmdline is None:
        print(
            "\nWARNING: this kernel was not started via `uv run`. "
            "BYOB launches Ray workers via the uv_runtime_env_hook, which only triggers under `uv run`. "
            "Restart Jupyter with: `uv run --extra byob jupyter lab`"
        )
    else:
        extras = []
        for i, tok in enumerate(uv_cmdline):
            if tok == "--extra" and i + 1 < len(uv_cmdline):
                extras.append(uv_cmdline[i + 1])
            elif tok.startswith("--extra="):
                extras.append(tok.split("=", 1)[1])
        all_extras = any(tok == "--all-extras" for tok in uv_cmdline)
        print(f"\nAncestor uv run cmdline: {uv_cmdline}")
        print(f"Detected extras: {extras or '(none)'}; --all-extras={all_extras}")
        if "byob" not in extras and not all_extras:
            raise RuntimeError(
                "Jupyter was started with `uv run` but WITHOUT `--extra byob` (or `--all-extras`). "
                "Ray's uv_runtime_env_hook will spin up workers in an environment that lacks "
                "nemo_curator, and BYOB will fail with `ModuleNotFoundError: nemo_curator` "
                "inside Ray. Shut down Jupyter and restart with: `uv run --extra byob jupyter lab`"
            )
        print("  ok  --extra byob is set on ancestor `uv run` (workers will inherit it)")
except ImportError:
    print("psutil unavailable, skipping ancestor uv-run check.")

ray_session_dir = Path("/tmp/ray")
stale = []
if ray_session_dir.exists() and shutil.which("ps"):
    out = subprocess.run(["ps", "-eo", "pid,cmd"], capture_output=True, text=True).stdout
    for line in out.splitlines():
        if "ray/_private/workers/default_worker.py" not in line and "raylet" not in line:
            continue
        if str(REPO_ROOT) in line:
            continue
        stale.append(line.strip())

if stale:
    print("\nWARNING: Ray processes from a different project appear to be running:")
    for line in stale[:5]:
        print(" ", line)
    print(
        "If BYOB later fails with `ModuleNotFoundError: nemo_curator` inside Ray, "
        "shut them down with `ray stop --force` (or kill the listed PIDs) and re-run."
    )
else:
    print("\nNo stale Ray workers detected from other projects.")

## 1c. Verify nemo-curator install

BYOB drives Curator for translation and semantic dedup. Confirm `nemo_curator` is importable from the kernel *and* that a few of its submodules used by BYOB (`modules`, `modifiers`, `filters`) load — those are what fail first when the install is partial.

In [ ]:
import importlib
from importlib.metadata import PackageNotFoundError, version

try:
    nc_version = version("nemo-curator")
except PackageNotFoundError:
    nc_version = "<not installed via package metadata>"

try:
    import nemo_curator
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "nemo_curator is not importable from this kernel. "
        "From the repo root run `uv sync --extra byob` and restart the Jupyter kernel."
    ) from exc

print(f"nemo_curator version : {nc_version}")
print(f"nemo_curator file    : {nemo_curator.__file__}")

# Submodules BYOB actually imports (see src/nemotron/steps/byob/runtime/*).
submodules = [
    "nemo_curator.pipeline",
    "nemo_curator.tasks",
    "nemo_curator.backends.ray_actor_pool",
    "nemo_curator.backends.ray_data",
    "nemo_curator.stages.text.embedders",
    "nemo_curator.stages.text.io.reader",
    "nemo_curator.stages.text.io.writer",
    "nemo_curator.stages.deduplication.semantic",
]
broken = []
for name in submodules:
    try:
        importlib.import_module(name)
        print(f"  ok   {name}")
    except Exception as exc:
        broken.append((name, exc))
        print(f"  FAIL {name} -> {exc.__class__.__name__}: {exc}")

if broken:
    raise RuntimeError(
        f"nemo_curator submodule import failed: {[b[0] for b in broken]}. "
        "The install is partial — re-run `uv sync --extra byob` and check for resolver errors."
    )

print("\nnemo_curator is correctly installed.")

## 2. Credentials

The default BYOB config uses NVIDIA-hosted OpenAI-compatible endpoints. Export `NGC_API_KEY` before starting Jupyter, or set it in this cell for the current notebook process. `NVIDIA_API_KEY` is also accepted. `HF_TOKEN` is optional for public Hugging Face datasets and embedding models.

In [ ]:
NGC_API_KEY = None
NVIDIA_API_KEY = None
HF_TOKEN = None

if NGC_API_KEY:
    os.environ["NGC_API_KEY"] = NGC_API_KEY
    os.environ.setdefault("NVIDIA_API_KEY", NGC_API_KEY)

if NVIDIA_API_KEY:
    os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY
    os.environ.setdefault("NGC_API_KEY", NVIDIA_API_KEY)

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ.setdefault("HUGGING_FACE_HUB_TOKEN", HF_TOKEN)

print("NGC_API_KEY set:", bool(os.environ.get("NGC_API_KEY")))
print("NVIDIA_API_KEY set:", bool(os.environ.get("NVIDIA_API_KEY")))
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")))

## 3. Create a finance source corpus

By default this cell writes small local finance notes so the notebook has a deterministic corpus. To pull live Wikipedia extracts instead, set `DOWNLOAD_FROM_WIKIPEDIA = True`.

In [ ]:
import json
import re
import time
import urllib.parse
import urllib.request

DOWNLOAD_FROM_WIKIPEDIA = False
WIKI_TITLES = ["Finance", "Financial_market", "Bond_(finance)", "Inflation"]


def clean_text(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def fetch_wikipedia_extract(title: str) -> str:
    params = urllib.parse.urlencode(
        {
            "action": "query",
            "format": "json",
            "prop": "extracts",
            "redirects": "1",
            "explaintext": "1",
            "titles": title,
        }
    )
    request = urllib.request.Request(
        f"https://en.wikipedia.org/w/api.php?{params}",
        headers={"User-Agent": "Nemotron-BYOB-example/1.0"},
    )
    with urllib.request.urlopen(request, timeout=60) as response:
        data = json.loads(response.read().decode("utf-8"))
    pages = data.get("query", {}).get("pages", {})
    for page in pages.values():
        return clean_text(page.get("extract", ""))
    return ""


sample_docs = {
    "finance_overview.txt": "Finance studies how people, firms, and governments allocate money over time. It includes saving, investing, borrowing, lending, budgeting, and risk management. Financial decisions compare expected returns, liquidity needs, and uncertainty.",
    "financial_markets.txt": "Financial markets connect savers and borrowers through instruments such as stocks, bonds, loans, and derivatives. Prices in liquid markets aggregate information about risk, expected cash flows, interest rates, and investor demand.",
    "bonds.txt": "A bond is a debt security. The issuer borrows principal from investors and typically promises periodic coupon payments plus principal repayment at maturity. Bond prices move inversely with market interest rates when other factors are unchanged.",
    "inflation.txt": "Inflation is a sustained increase in the general price level. It reduces purchasing power when wages or investment returns do not keep pace. Central banks often use interest-rate policy to influence inflation and economic activity.",
}

if DOWNLOAD_FROM_WIKIPEDIA:
    for title in WIKI_TITLES:
        text = fetch_wikipedia_extract(title)
        if text:
            (ASSETS_DIR / f"{title}.txt").write_text(f"# {title}\n\n{text}\n", encoding="utf-8")
        time.sleep(0.35)
else:
    for filename, text in sample_docs.items():
        (ASSETS_DIR / filename).write_text(text + "\n", encoding="utf-8")

for path in sorted(ASSETS_DIR.glob("*.txt")):
    print(path.name, path.stat().st_size, "bytes")

## 4. Write the example config from the packaged BYOB config

This keeps the notebook consistent with the framework: the packaged config is the template, and the example only overrides paths, run name, and small-run controls.

In [ ]:
cfg = yaml.safe_load(STEP_DEFAULT_CONFIG.read_text(encoding="utf-8"))

cfg.update(
    {
        "expt_name": "finance_wiki_mcq",
        "input_dir": str(EXAMPLE_DIR / "assets"),
        "output_dir": str(OUTPUT_DIR),
        "language": "en-US",
        "source_subjects": ["management", "marketing"],
        "target_source_mapping": {
            "wiki_finance": {
                "subjects": {
                    "management": 0.5,
                    "marketing": 0.5,
                }
            }
        },
        "few_shot_samples_per_query": 1,
        "queries_per_target_subject_document": 1,
        "num_questions_per_query": 1,
        "ndd_batch_size": 4,
        "do_coverage_check": False,
        "remove_hallucinated": True,
        "remove_easy": False,
    }
)
cfg["coverage_check_config"] = {"model_identifier": None, "window_size": None}

WORKING_CONFIG.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
print(WORKING_CONFIG.read_text(encoding="utf-8"))

## 5. Run BYOB pipeline

In [ ]:
def run_nemotron_byob(*args: str, check: bool = True) -> subprocess.CompletedProcess[str]:
    env = os.environ.copy()
    existing_pythonpath = env.get("PYTHONPATH", "")
    env["PYTHONPATH"] = str(SRC_DIR) + (os.pathsep + existing_pythonpath if existing_pythonpath else "")
    cmd = [sys.executable, "-m", "nemotron", "byob", *map(str, args)]
    print("$", " ".join(cmd))
    return subprocess.run(cmd, cwd=REPO_ROOT, env=env, text=True, check=check)


run_nemotron_byob("--list-families")

The next cell defaults to preview mode. Set `RUN_BYOB = True` to run `prepare` and `generate`.

In [ ]:
RUN_BYOB = True

prepare_args = ["--family", "mcq", "--stage", "prepare", "--config", str(WORKING_CONFIG)]
generate_args = ["--family", "mcq", "--stage", "generate", "--config", str(WORKING_CONFIG)]

if RUN_BYOB:
    if not (os.environ.get("NGC_API_KEY") or os.environ.get("NVIDIA_API_KEY")):
        raise RuntimeError("NGC_API_KEY or NVIDIA_API_KEY is required before running BYOB generation.")
    run_nemotron_byob(*prepare_args)
    run_nemotron_byob(*generate_args)
else:
    print("Preview only. Commands to run:")
    print("python -m nemotron byob", " ".join(prepare_args))
    print("python -m nemotron byob", " ".join(generate_args))

## 6. Preview the outputs

In [ ]:
import json

import pandas as pd
from IPython.display import display

benchmark_dir = OUTPUT_DIR / cfg["expt_name"]
raw_benchmark = benchmark_dir / "benchmark_raw.parquet"
final_benchmark = benchmark_dir / "benchmark.parquet"

print("Raw benchmark:", raw_benchmark, raw_benchmark.exists())
print("Final benchmark:", final_benchmark, final_benchmark.exists())

if final_benchmark.exists():
    df = pd.read_parquet(final_benchmark)
    print(df.columns.tolist())
    display(df.head())
else:
    print("Run the BYOB prepare/generate cell first, or point final_benchmark at an existing output.")

## 7. Optional translation config

This mirrors the packaged translation config and points it at the generated `benchmark.parquet`.

In [ ]:
translate_cfg = yaml.safe_load(STEP_TRANSLATE_CONFIG.read_text(encoding="utf-8"))
translate_cfg.update(
    {
        "expt_name": "finance_wiki_mcq_hi",
        "dataset_path": str(final_benchmark),
        "output_dir": str(OUTPUT_DIR),
        "source_language": "en-US",
        "target_language": "hi-IN",
    }
)
TRANSLATION_CONFIG.write_text(yaml.safe_dump(translate_cfg, sort_keys=False), encoding="utf-8")
print(TRANSLATION_CONFIG.read_text(encoding="utf-8"))


## 7. Optional translation config

Set `RUN_TRANSLATION = True` after the MCQ benchmark exists.

In [ ]:

RUN_TRANSLATION = True
translation_args = ["--family", "mcq", "--stage", "translate", "--config", str(TRANSLATION_CONFIG)]

if RUN_TRANSLATION:
    if not final_benchmark.exists():
        raise RuntimeError("Generate benchmark.parquet before running translation.")
    if not (os.environ.get("NGC_API_KEY") or os.environ.get("NVIDIA_API_KEY")):
        raise RuntimeError("NGC_API_KEY or NVIDIA_API_KEY is required before running BYOB translation.")
    run_nemotron_byob(*translation_args)
else:
    print("Preview only. Command to run:")
    print("python -m nemotron byob", " ".join(translation_args))